# Data cleaning: Tourist_Accommodation22092025

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [23]:
# Importamos el CSV
input_path = Path("data") / "staySpain_raw_22092025.csv"
df = pd.read_csv(input_path)

In [3]:
# Hacemos una copia de df para no modificar el df original
df_clean = df.copy()

In [4]:
# Info de df_clean
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 35 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   apartment_id                 8000 non-null   int64  
 1   name                         7997 non-null   object 
 2   description                  7946 non-null   object 
 3   host_id                      8000 non-null   int64  
 4   neighbourhood_name           8000 non-null   object 
 5   neighbourhood_district       4861 non-null   object 
 6   room_type                    8000 non-null   object 
 7   accommodates                 8000 non-null   int64  
 8   bathrooms                    7957 non-null   float64
 9   bedrooms                     7961 non-null   float64
 10  beds                         7992 non-null   float64
 11  amenities_list               7983 non-null   object 
 12  price                        7829 non-null   float64
 13  minimum_nights    

In [5]:
# Visualizamos todas las columnas en el output
pd.set_option("display.max_columns", None)

In [6]:
# Comprobamos si hay filas enteras duplicadas
df_clean.duplicated().any()

np.False_

In [7]:
# Comprobamos si hay duplicados en "apartment_id"
df_clean.duplicated(subset="apartment_id").sum()

np.int64(307)

In [8]:
# Buscamos duplicados en "apartment_id"
duplicados = df_clean[df_clean.duplicated("apartment_id", keep=False)]

In [9]:
# Ordenamos los registros para eliminar los duplicados con "insert_date" más antigua
df_clean = df_clean.sort_values("insert_date", ascending=False)\
                     .drop_duplicates("apartment_id", keep='first')\
                     .sort_index()

In [10]:
# Capitalizamos la primera letra de los valores en la columna "city" y "country"
df_clean["city"] = df_clean["city"].str.capitalize()
df_clean["country"] = df_clean["country"].str.capitalize()

In [11]:
df_clean.dtypes

apartment_id                     int64
name                            object
description                     object
host_id                          int64
neighbourhood_name              object
neighbourhood_district          object
room_type                       object
accommodates                     int64
bathrooms                      float64
bedrooms                       float64
beds                           float64
amenities_list                  object
price                          float64
minimum_nights                   int64
maximum_nights                   int64
has_availability                object
availability_30                  int64
availability_60                  int64
availability_90                  int64
availability_365                 int64
number_of_reviews                int64
first_review_date               object
last_review_date                object
review_scores_rating           float64
review_scores_accuracy         float64
review_scores_cleanliness

In [12]:
# Conversión de datos a date en las columnas que muestran fechas
df_clean["first_review_date"] = pd.to_datetime(df_clean['first_review_date'], errors='coerce', format='%d/%m/%Y')
df_clean["last_review_date"] = pd.to_datetime(df_clean['last_review_date'], errors='coerce', format='%d/%m/%Y')
df_clean["insert_date"] = pd.to_datetime(df_clean['insert_date'], errors='coerce', format='%d/%m/%Y')

In [13]:
# Conversión de float a int
df_clean["bathrooms"] = df_clean['bathrooms'].astype("Int64")
df_clean["bedrooms"] = df_clean['bedrooms'].astype("Int64")
df_clean["beds"] = df_clean['beds'].astype("Int64")

In [ ]:
# Creamos una columna nueva para convertir "is_instant_bookable" en booleano
df_clean["is_instant_bookable_bool"] = df_clean["is_instant_bookable"].replace({"VERDADERO": True, "FALSO": False}).astype(bool)

df_clean.head()

In [ ]:
# Eliminamos la columna original
df_clean = df_clean.drop(columns="is_instant_bookable")

# Renombramos la nueva columna
df_clean = df_clean.rename({"is_instant_bookable_bool":"is_instant_bookable"}, axis=1)

df_clean.head()

In [16]:
# # Repetimos proceso con has_availability pero con map() para mantener los NaN
mapping={"VERDADERO":True, "FALSO":False}
df_clean["has_availability"] = df_clean["has_availability"].map(mapping)


In [17]:
df_clean["has_availability"].isna().sum()

np.int64(534)

In [18]:
# Imputación de nombre para registros con NaN en "name"
df_clean['name'] = df_clean['name'].fillna(df_clean['room_type'] + ' ' + df_clean['neighbourhood_name'])

In [19]:
# Imputación de descripción para registros con NaN en "description"
df_clean['description'] = df_clean['description'].fillna(df_clean['name'])